<a href="https://colab.research.google.com/github/rendzina/BigDataAndVisualisation/blob/main/Colab/EnvironmentalAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environmental API JSON data: Spark example
*MKU: Big Data and Visualisation*

## Reading data from an IoT Photon board with BME680 sensor

An environmental sensor in an office at Cranfield logs to [ThingSpeak](https://thingspeak.com). ThingSpeak exposes REST endpoints for the channel. This notebook requests **hourly averages for the last 24 hours** (`minutes=1440`, `average=60`), parses the JSON in Python, builds a **Spark DataFrame**, then summarises and plots the series.

# Useful links
## Data source
https://thingspeak.com/channels/783513/api_keys

## Programming
https://dev.to/wachuka_james/building-a-weather-data-pipeline-with-pyspark-prefect-and-google-cloud-19k8
https://phoenixnap.com/kb/spark-streaming

## Start PySpark (Colab)

Google Colab already includes PySpark and a Java runtime, so you only need to **import** and create a `SparkSession`. No separate Spark download or `findspark` step is required.

The next cell builds a local Spark session (`local[*]`) named `environmentalData`. That `spark` object is used to create DataFrames from in-memory rows after the API call.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("environmentalData")
    .master("local[*]")
    .getOrCreate()
)
spark


## Load data from the API

ThingSpeak returns JSON with a `channel` block (metadata and field labels) and a `feeds` array (one object per time bucket). The next cells set the request URL, then define a function that downloads the payload, reads field names from the channel definition, and unions one small DataFrame per feed row so column names stay aligned with the API.

### API request URL

Adjust `api_url` if you use another channel or time window. The commented block shows the typical JSON shape so you can map `field1`…`field4` to temperature, humidity, and derived quantities.

In [ ]:
# Sensor channel: hourly averages over the last 24 hours
api_url = "https://api.thingspeak.com/channels/760368/feeds.json?minutes=1440&average=60"

# Example JSON shape (abbreviated):
# {
#   "channel": { "field1": "Temperature", "field2": "Humidity", ... },
#   "feeds": [
#     { "created_at": "2024-05-27T08:00:00Z", "field1": "24.04", ... },
#     ...
#   ]
# }


### Inspect the raw JSON response

Before parsing the payload into a Spark DataFrame, it is helpful to see what the API actually returns. The next cell defines a small helper that calls the endpoint and returns the decoded JSON, then pretty-prints it so you can see the `channel` metadata block and the `feeds` array side by side.

In [ ]:
import json
import requests


def fetch_environmental_data_json():
    response = requests.get(api_url)
    return response.json()


print(json.dumps(fetch_environmental_data_json(), indent=2))

### Build the DataFrame

`fetch_environmental_data()` calls `requests.get`, parses JSON, builds a `StructType` using the channel’s field labels, then appends each feed row with `union`. Using `float(... or 0)` avoids errors if a field is missing in a single feed.

In [ ]:
# Fetch ThingSpeak JSON and build a Spark DataFrame
from pyspark.sql.types import FloatType, StructField, StructType, TimestampType
import datetime
import requests


def fetch_environmental_data():
    response = requests.get(api_url)
    data = response.json()

    field1_name = data["channel"]["field1"]
    field2_name = data["channel"]["field2"]
    field3_name = data["channel"]["field3"]
    field4_name = data["channel"]["field4"]

    schema = StructType(
        [
            StructField("created_at", TimestampType(), True),
            StructField(field1_name, FloatType(), True),
            StructField(field2_name, FloatType(), True),
            StructField(field3_name, FloatType(), True),
            StructField(field4_name, FloatType(), True),
        ]
    )
    df = spark.createDataFrame([], schema)

    for items in data["feeds"]:
        new_row = spark.createDataFrame(
            [
                (
                    datetime.datetime.strptime(
                        items["created_at"], "%Y-%m-%dT%H:%M:%SZ"
                    ),
                    float(items["field1"] or 0),
                    float(items["field2"] or 0),
                    float(items["field3"] or 0),
                    float(items["field4"] or 0),
                )
            ],
            schema,
        )
        df = df.union(new_row)
    return df


### Run the fetch and inspect

`show()` prints a sample; `count()` should match the number of hourly buckets returned (up to 24 for this URL). `columns` and `dtypes` confirm names and types after parsing.

In [ ]:
environmental_df = fetch_environmental_data()

environmental_df.show()
print(environmental_df.count())
print(environmental_df.columns)
print(environmental_df.dtypes)


# Analyses
## Basic statistics

The next cell does two things: it moves a **small** result to **pandas** for `describe()`, and it copies the four numeric columns into an **RDD** to compute **MLlib** column statistics. That pattern mirrors introductory Spark ML workflows (MLlib RDD API is legacy but still useful for teaching summary stats).

In [ ]:
import numpy as np
import pyspark.mllib.stat as st
from pyspark.sql.functions import date_format

envdata = environmental_df.withColumn(
    "created_at", date_format("created_at", "yyyy-MM-dd HH:mm:ss")
).toPandas()
print(envdata.describe())

rdd = (
    environmental_df.select(
        "Temperature", "Humidity", "Dew_point", "Heat_Index"
    )
    .rdd.map(lambda row: [e for e in row])
)
print(rdd.collect())

mllib_stats = st.Statistics.colStats(rdd)

print("Metric\t\tMean\tStd dev")
for col, mean, var in zip(
    ["Temperature", "Humidity", "Dew_point", "Heat_Index"],
    mllib_stats.mean(),
    mllib_stats.variance(),
):
    print("{0}:\t{1:.2f}\t{2:.2f}".format(col, mean, np.sqrt(var)))


**Driver memory:** `toPandas()` pulls the full DataFrame into the notebook process. That is acceptable here because the series is short; for large datasets, keep aggregations in Spark and only collect summaries or samples.

# Charts
## Bar chart: temperature over time

Each plotting cell converts timestamps to strings for the axis, then builds the chart in matplotlib. Run the earlier cells so `environmental_df` exists.

In [ ]:
import matplotlib.pyplot as plt
from pyspark.sql.functions import date_format

envdata = environmental_df.withColumn(
    "created_at", date_format("created_at", "yyyy-MM-dd HH:mm:ss")
).toPandas()

plt.figure(1).set_figwidth(15)
plt.title("Office temperature over time")
plt.xlabel("Time recorded")
plt.xticks(rotation=90)
plt.ylabel("Temperature (°C)")
plt.bar(envdata["created_at"], envdata["Temperature"])
plt.show()


## Grouped bar chart: all series

Pandas `plot` draws temperature, humidity, dew point, and heat index side by side for each timestamp.

In [ ]:
import matplotlib.pyplot as plt
from pyspark.sql.functions import date_format

envdata = environmental_df.withColumn(
    "created_at", date_format("created_at", "yyyy-MM-dd HH:mm:ss")
).toPandas()

envdata.plot(
    x="created_at",
    kind="bar",
    stacked=False,
    figsize=(14, 8),
    title="Office conditions over the past 24 hours",
)
plt.show()


## Line chart: temperature

Same data as a line plot using pandas on a shared matplotlib axes.

In [ ]:
import matplotlib.pyplot as plt
from pyspark.sql.functions import date_format

envdata = environmental_df.withColumn(
    "created_at", date_format("created_at", "yyyy-MM-dd HH:mm:ss")
).toPandas()

plt.figure().set_figwidth(15)
axes = plt.axes()
plt.title("Office temperature over time")
plt.xlabel("Time recorded")
plt.ylabel("Temperature (°C)")
axes.xaxis.set_tick_params(rotation=90)
envdata.plot(
    x="created_at", y="Temperature", legend=True, kind="line", ax=axes
)
plt.show()
